In [1]:
import pandas as pd
import numpy as np
import ast
from sentence_transformers import SentenceTransformer
from datetime import datetime, date
import json

/home/bsc/bsc093754/miniforge3/envs/datamap_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('/home/bsc/bsc093754/GIT/social-media-data-map/data/processed/aggregate_paths_2026-05-14.csv')
df = df.sample(n=100)

ROOT = '/home/bsc/bsc093754/GIT/social-media-data-map'
save_dir = f'{ROOT}/data/processed'

In [3]:
def combine_list(df):

    df['final_path'] = None

    for ix, row in df.iterrows():
        if row['platform'] == 'Tiktok':
            final_path =  row['path']
            
        else:
            file_list = row['file_path'].split('/')
            path_list = row['path']
        
            
            if isinstance(path_list, str):
                path_list = ast.literal_eval(path_list)

            if isinstance(path_list, list):
                final_path = file_list + path_list
                path = '/'.join(path_list)


        final_path = '/'.join(final_path)

        df.at[ix, 'final_path'] = final_path 
        df.at[ix, 'path'] = path       
    
    return df
    


In [4]:
def unique_paths(df):
    platforms = df['platform'].unique()
    df_unique = None
    for p in platforms:
        df_red =  df[df['platform'] == p]
        df_red['final_path'].unique()
        if df_unique is None:
            df_unique = df_red
            
        else:
            df_unique = pd.concat([df_unique, df_red], axis=0, ignore_index=True)
           
    df_unique = df_unique[['platform', 'final_path', 'path', 'json_name']]
    return df_unique

In [5]:
def embed_lists(model, df):
    model = SentenceTransformer(model)
    df['final_path_emb'] = None
    df['path_emb'] = None
    df['json_name_emb'] = None
    
    columns = ['final_path', 'path', 'json_name']
    for c in columns:
        for ix, row in df.iterrows():
            list_path = row[f'{c}']
            print(list_path)
            
            try:
                em = model.encode(list_path)
                em = json.dumps(em.tolist())
                df.at[ix, f'{c}_emb'] = em
            except: 
                df.at[ix, f'{c}_emb'] = np.nan

    df.to_csv(f'{save_dir}/path_embeddings_{date.today()}.csv')
    return df
        



In [6]:
def embed_corpus(df, model):
    model = SentenceTransformer(model)
    

    platforms = ['Tiktok', 'Facebook', 'Instagram', 'Twitter', 'Youtube']
    nodes = []
    print(df.columns)
    df = df.replace(np.nan, 'MISSING')
    for p in platforms:
        corpus_final_path = []

        if p == 'Tiktok':
                corpus_final_path = df['final_path'].tolist()

                node = {'platform': 'Tiktok',
                       'corpus_final_path': model.encode(corpus_final_path, show_progress_bar=True, convert_to_tensor=True)}
        
        else:
                
                corpus_final_path = df['final_path'].tolist()
                corpus_path = df['path'].tolist()
                corpus_json_name = df['json_name'].tolist()

                print(corpus_json_name)

                node = {'platform': p,
                        'corpus_final_path': model.encode(corpus_final_path, show_progress_bar=True).tolist(),
                        'corpus_path': model.encode(corpus_path, show_progress_bar=True).tolist(),
                        'corpus_json_name': model.encode(corpus_json_name , show_progress_bar=True).tolist()}
                
        nodes.append(node)


    json_str = json.dumps(nodes, indent=2)
    with open(f'{save_dir}/corpus_embeddings_{date.today()}.json', "w") as f:
            f.write(json_str)

    return nodes





In [7]:
model = "/gpfs/projects/bsc100/models/sentence-transformers/all-MiniLM-L6-v2"
df = combine_list(df)
df = unique_paths(df)

In [8]:

node = embed_corpus(df, model)
print(json.dumps(node, indent=2))


/home/bsc/bsc093754/miniforge3/envs/datamap_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9956.98it/s]


Index(['platform', 'final_path', 'path', 'json_name'], dtype='str')


Batches: 100%|██████████| 4/4 [00:00<00:00, 11.45it/s]


['stories.json', 'recently_unfollowed_profiles.json', 'last_known_location.json', 'login_activity.json', 'stories.json', 'reels_comments.json', 'saved_collections.json', 'avatar_items.json', 'message_1.json', 'last_known_location.json', 'camera_information.json', 'instagram_friend_map.json', 'audience_insights.json', 'stories.json', 'personal_information.json', 'liked_posts.json', 'stories.json', 'emoji_sliders.json', 'content_interactions.json', "profiles_you're_not_interested_in.json", 'followers.json', 'stories.json', 'signup_details.json', 'recommended_topics.json', 'posts.json', 'profile_searches.json', 'message_1.json', 'other_categories_used_to_reach_you.json', 'personal_information.json', 'locations_of_interest.json', 'personal_information.json', "information_you've_submitted_to_advertisers.json", 'polls.json', 'your_scheduled_chat_notifications.json', 'your_muted_story_teaser_creators.json', 'posts.json', 'instagram_profile_information.json', 'eligibility.json', 'personal_info

Batches: 100%|██████████| 4/4 [00:00<00:00, 72.42it/s]


...

['stories.json', 'recently_unfollowed_profiles.json', 'last_known_location.json', 'login_activity.json', 'stories.json', 'reels_comments.json', 'saved_collections.json', 'avatar_items.json', 'message_1.json', 'last_known_location.json', 'camera_information.json', 'instagram_friend_map.json', 'audience_insights.json', 'stories.json', 'personal_information.json', 'liked_posts.json', 'stories.json', 'emoji_sliders.json', 'content_interactions.json', "profiles_you're_not_interested_in.json", 'followers.json', 'stories.json', 'signup_details.json', 'recommended_topics.json', 'posts.json', 'profile_searches.json', 'message_1.json', 'other_categories_used_to_reach_you.json', 'personal_information.json', 'locations_of_interest.json', 'personal_information.json', "information_you've_submitted_to_advertisers.json", 'polls.json', 'your_scheduled_chat_notifications.json', 'your_muted_story_teaser_creators.json', 'posts.json', 'instagram_profile_information.json', 'eligibility.json', 'personal_info

Batches: 100%|██████████| 4/4 [00:00<00:00, 71.74it/s]


['stories.json', 'recently_unfollowed_profiles.json', 'last_known_location.json', 'login_activity.json', 'stories.json', 'reels_comments.json', 'saved_collections.json', 'avatar_items.json', 'message_1.json', 'last_known_location.json', 'camera_information.json', 'instagram_friend_map.json', 'audience_insights.json', 'stories.json', 'personal_information.json', 'liked_posts.json', 'stories.json', 'emoji_sliders.json', 'content_interactions.json', "profiles_you're_not_interested_in.json", 'followers.json', 'stories.json', 'signup_details.json', 'recommended_topics.json', 'posts.json', 'profile_searches.json', 'message_1.json', 'other_categories_used_to_reach_you.json', 'personal_information.json', 'locations_of_interest.json', 'personal_information.json', "information_you've_submitted_to_advertisers.json", 'polls.json', 'your_scheduled_chat_notifications.json', 'your_muted_story_teaser_creators.json', 'posts.json', 'instagram_profile_information.json', 'eligibility.json', 'personal_info

Batches: 100%|██████████| 4/4 [00:00<00:00, 69.00it/s]


TypeError: Object of type Tensor is not JSON serializable

In [ ]:
df = embed_lists(model, df)